# Chapter 1

### Parameters in a neural network

<center><img src="images/01.01.png"  style="width: 400px, height: 300px;"/></center>

# Chapter 2

### Multi label vs multi class

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>


### Can we solve multi-class problem without using softmax?

- use 'sigmoid' activation function in the output layer. This will act as one-vs-rest classification (2-class classification) problem
- Since this becomes a two-class classification problem now, use 'binary_crossentropy' as the loss function
- You can use 'adam' as optimizer
- This approach is preferable for multi-label classification problem instead of multi-class classification problem

<center><img src="images/02.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.03.png"  style="width: 400px, height: 300px;"/></center>


### Keras Neural Network

```
data = pd.read_csv('dataset.csv')
sns.pairplot(data, hue='target')  # Good to explore the dataset
plt.show()
X = data.drop(['target'], axis=1).values 
n_cols = X.shape[1] # Get the number of features other than target column
# For categorical target variable, you need something like one-hot-encoding
from tensorflow.keras.utils import to_categorical
y = to_categorical(data['target']) # Use this for one-hot-encoding if target is a class and it is a classification problem
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
def create_model(optimizer='adam', activation='relu', nl=1,nn=256):
  model = Sequential()
  # for i in range(nl): # you can also automate this process of creating layers and neurons with specified arguments
  # # Layers have nn neurons
  #   model.add(Dense(nn, activation=activation))
  # Start with the first hidden layer, where information comes in shape (dataset column, any number of rows)
  model.add(Dense(100, activation=activation, input_shape = (n_cols,))) # First hidden layer, where values directly come from dataset of n features, with unknown number of datapoints
  model.add(Dense(100, activation=activation, kernel_initializer='normal')) # Second hidden layer
  model.add(Dense(100, activation=activation)) # Third hidden layer
  from tensorflow.keras.layers import BatchNormalization
  model.add(BatchNormalization()) # Add batch normalization for the outputs of the layer above
  # use 'softmax' for multi-class classification, 'sigmoid' for binary or multi-label classification
  model.add(Dense(3, activation='softmax')) # Output layer, 3 nodes for 3 class prediction
  my_optimizer = SGD(lr=lr) # # You can also use custom optimizer like this
  # For loss function, use 'categorical_crossentropy' for multiclass, 'binary_crossentropy' for binary or multi-label, 'mean_squared_error'  for regression
  model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy']) # optimizer = my_optimizer
  return model
# K-fold is costly for neural network, so use validation data as a wise choice to get best validation score with early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_monitor = EarlyStopping(monitor='val_loss', patience=2) # Terminates once validation loss stops improving
# Instantiate a model checkpoint callback, this will automatically save the model when best result is produced
from keras.callbacks import ModelCheckpoint
model_save = ModelCheckpoint('best_model.hdf5', save_best_only=True)
# When fitting, you can use validation data using : "validation_data=(X_test, y_test)" or directly use validation_split
model = create_model()
model_1_training = model.fit(X_train, y_train, validation_split=0.3, epochs=20, batch_size=128, callbacks = [early_stopping_monitor, model_save], verbose=0)
predictions = model.predict(X_test)
probability_true = predictions[:,1] 

from tensorflow.keras.models import load_model
init_weights = model.get_weights() # Save model weights
model.save('model_file.h5') # Saving  model
my_model = load_model('model_file.h5') # loading model
my_model.summary() # See model summary
history = model_1_training # Extracting fitting history
print(history.params) # See all parameters
print(history.history.keys()) # See what you can extract, eg :  val_loss = history.history['val_loss'], val_accuracy = history.history['val_accuracy']
accuracy = model.evaluate(X_test, y_test)[1] # Evaluate
preds = model.predict(test_set) # Predict 
# Extract the position of highest probability from each pred vector (For classification of multi-class problem)
preds_chosen = [np.argmax(pred) for pred in preds]

# Helper function to visualize loss learning curve (as epochs go by, loss should decrese)
def plot_loss(loss,val_loss): # This function also helps us to identify overfitting by looking at training loss vs testing loss
  plt.figure()
  plt.plot(loss)
  plt.plot(val_loss)
  plt.title('Model loss')
  plt.ylabel('Loss')
  plt.xlabel('Epoch')
  plt.legend(['Train', 'Test'], loc='upper right')
  plt.show()

# Helper function to visualize accuracy learning curve (as epochs go by, accuracy should increase)
def plot_accuracy(acc,val_acc): 
  # Plot training & validation accuracy values with the same function as "plot_loss" ........
  plt.figure()

h_callback = model.fit(X_train, y_train, epochs = 25, validation_data=(X_test, y_test)) # Train your model and save its history
plot_loss(h_callback.history['loss'], h_callback.history['val_loss']) # Plot train vs test loss during training
plot_accuracy(h_callback.history['accuracy'], h_callback.history['val_accuracy']) # Plot train vs test accuracy during training

# Visualize if train size increase accuracy
train_accs = []
tests_accs = []
train_sizes = [0.2, 0.4, 0.6]
init_weights = model.get_weights()
for train_size in train_sizes:
  X_train_frac, _, y_train_frac, _ = train_test_split(X_train, y_train, train_size=train_size) # Split a fraction according to train_size
  model.set_weights(init_weights) # Make sure to use same random weights 
  model.fit(X_train_frac, y_train_frac, epochs=100, verbose=0, callbacks=[EarlyStopping(monitor='loss', patience=1)]) # Fit model on the training set fraction
  train_acc = model.evaluate(X_train_frac, y_train_frac, verbose=0)[1] # Get the accuracy for this training set fraction
  train_accs.append(train_acc)
  test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
  test_accs.append(test_acc)
plt.plot(train_accs)
plt.plot(test_accs)

# k-fold Cross validation
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier # Import sklearn wrapper from keras
model = KerasClassifier(build_fn=create_model, epochs=6, batch_size=16) # Create a model as a sklearn estimator
from sklearn.model_selection import cross_val_score
kfold = cross_val_score(model, X, y, cv=5) # Check how your keras model performs with 5 fold crossvalidation
kfold.mean() # Print the mean accuracy per fold
# Fine-tuning with RandomSearchCV
params = dict(optimizer=['sgd', 'adam'], epochs=3, batch_size=[5, 10, 20], activation=['relu','tanh'], nl=[1, 2, 9], nn=[128,256,1000])
random_search = RandomizedSearchCV(model, params_dist=params, cv=3)
random_search_results = random_search.fit(X, y)
random_search_results.best_score_
random_search_results.best_params_
```

# Chapter 3

### Visualizing overfitting

<center><img src="images/03.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.03.png"  style="width: 400px, height: 300px;"/></center>

### Unstable Learning curve

<center><img src="images/03.02.png"  style="width: 400px, height: 300px;"/></center>


### Which Activation Functions to use

- ReLU are a good first choice
- Sigmoids not recommended for deep models
- Tune with experimentation

<center><img src="images/03.04.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.06.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.07.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.08.png"  style="width: 400px, height: 300px;"/></center>

### Batch

- batch = training set
- mini-batch = subset of training set
- for each mini-batch, weight in each epoch is updated once
- helps faster training, less ram usage
- noise helps to reduce error and escape local minima
- Disadvantages: more iterations, needs adjusted batch size
<center><img src="images/03.09.png"  style="width: 400px, height: 300px;"/></center>


# Batch normalization

- helps avoid problems of activation functions and gradients
- makes sure inputs of the next layers are normalized
- Improves gradient .ow
- Allows higher learning rates
- Reduces dependence on weight initializations
- Acts as an unintended form of regularization
- Limits internal covariate shift
<center><img src="images/03.10.png"  style="width: 400px, height: 300px;"/></center>

